In [ ]:

# ================================================================
# MAHARASHTRA WHEAT AND MAIZE PRICE DATASET
# 2015-01-01 to 2026-11-19
# District-level daily aggregation
#
# Source: Third-party AGMARKNET historical data compilation
# Processing: DuckDB + Pandas
# ================================================================

import os
import re
import json
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import duckdb

from huggingface_hub import HfApi, hf_hub_download
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ================================================================
# 1. PROJECT CONFIGURATION
# ================================================================

REPO_ID = (
    "DHIVAKARNAIDUSUNKARI/"
    "mandi-india-daily-prices-2001-2026"
)

START_DATE = "2015-01-01"
END_DATE = "2026-11-19"

COMMODITIES = ["WHEAT", "MAIZE"]

# Automatically use Documents folder.
BASE_DIR = Path.home() / "Documents" / "PROJECT 4"

OUTPUT_DIR = BASE_DIR / "output" / "mandi_prices"
DOWNLOAD_DIR = OUTPUT_DIR / "raw"
LOG_DIR = OUTPUT_DIR / "logs"

for folder in [OUTPUT_DIR, DOWNLOAD_DIR, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("=" * 65)
print("PROJECT CONFIGURATION")
print("=" * 65)

print("Base directory:", BASE_DIR)
print("Output directory:", OUTPUT_DIR)
print("Download directory:", DOWNLOAD_DIR)

# ================================================================
# 2. CONNECT TO DATASET REPOSITORY
# ================================================================

print("\nConnecting to dataset repository...")

api = HfApi()

all_files = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type="dataset"
)

parquet_files = [
    f for f in all_files
    if f.lower().endswith(".parquet")
]

print("Total repository files:", len(all_files))
print("Parquet files:", len(parquet_files))

if not parquet_files:
    raise RuntimeError(
        "No Parquet files found. Check the dataset repository."
    )

# ================================================================
# 3. DOWNLOAD PARQUET FILES
# ================================================================

# Download all available Parquet shards.
# If the repository is very large, inspect the file listing first.

local_files = []

for file in tqdm(parquet_files, desc="Downloading data"):

    local_path = hf_hub_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        filename=file,
        local_dir=str(DOWNLOAD_DIR)
    )

    local_files.append(str(local_path))

print("\nDownloaded files:", len(local_files))

# ================================================================
# 4. CONNECT DUCKDB
# ================================================================

con = duckdb.connect()

# Escape file paths for SQL.
file_list = "[" + ",".join(
    "'" + p.replace("\\", "/").replace("'", "''") + "'"
    for p in local_files
) + "]"

# ================================================================
# 5. INSPECT ACTUAL DATASET SCHEMA
# ================================================================

print("\nInspecting dataset schema...")

schema = con.execute(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        {file_list},
        union_by_name=true
    )
    """
).df()

print(schema.to_string(index=False))

# Save schema.
schema.to_csv(
    LOG_DIR / "dataset_schema.csv",
    index=False
)

# ================================================================
# 6. AUTOMATIC COLUMN MAPPING
# ================================================================

# Column names differ between datasets.
# Map known variants to standard names.

available_columns = {
    str(c).strip().lower(): str(c)
    for c in schema["column_name"]
}

column_aliases = {
    "State": [
        "state", "state_name"
    ],
    "District": [
        "district", "district_name"
    ],
    "Market": [
        "market", "market_name", "mandi"
    ],
    "Commodity": [
        "commodity", "commodity_name"
    ],
    "Variety": [
        "variety", "variety_name"
    ],
    "Grade": [
        "grade", "quality"
    ],
    "Date": [
        "arrival_date", "date", "reported_date"
    ],
    "Min_Price": [
        "min_price", "minimum_price"
    ],
    "Max_Price": [
        "max_price", "maximum_price"
    ],
    "Modal_Price": [
        "modal_price", "modalprice"
    ],
    "Arrivals": [
        "arrivals", "arrival", "arrival_quantity"
    ]
}

column_map = {}

for standard_name, aliases in column_aliases.items():

    for alias in aliases:

        if alias.lower() in available_columns:
            column_map[standard_name] = (
                available_columns[alias.lower()]
            )
            break

print("\nDetected column mapping:")

for key, value in column_map.items():
    print(f"{key}: {value}")

required_columns = [
    "State",
    "District",
    "Market",
    "Commodity",
    "Date",
    "Modal_Price"
]

missing_columns = [
    col for col in required_columns
    if col not in column_map
]

if missing_columns:
    raise ValueError(
        "Required columns not detected: "
        + str(missing_columns)
        + "\nPlease inspect dataset_schema.csv and update "
          "column_aliases."
    )

# ================================================================
# 7. BUILD SQL QUERY
# ================================================================

def sql_identifier(name):
    return '"' + name.replace('"', '""') + '"'

def sql_column(standard_name, optional=False):

    if standard_name not in column_map:

        if optional:
            return "NULL"

        raise ValueError(
            f"Missing required field: {standard_name}"
        )

    return sql_identifier(column_map[standard_name])


select_fields = [
    f"{sql_column('State')} AS State",
    f"{sql_column('District')} AS District",
    f"{sql_column('Market')} AS Market",
    f"{sql_column('Commodity')} AS Commodity",
    f"{sql_column('Variety', True)} AS Variety",
    f"{sql_column('Grade', True)} AS Grade",
    f"TRY_CAST({sql_column('Date')} AS DATE) AS Arrival_Date",
    f"TRY_CAST({sql_column('Min_Price', True)} AS DOUBLE) AS Min_Price",
    f"TRY_CAST({sql_column('Max_Price', True)} AS DOUBLE) AS Max_Price",
    f"TRY_CAST({sql_column('Modal_Price')} AS DOUBLE) AS Modal_Price",
    f"TRY_CAST({sql_column('Arrivals', True)} AS DOUBLE) AS Arrivals"
]

query = f"""
SELECT
    {', '.join(select_fields)}
FROM read_parquet(
    {file_list},
    union_by_name=true
)
WHERE
    UPPER(TRIM(CAST({sql_column('State')} AS VARCHAR)))
        = 'MAHARASHTRA'

    AND UPPER(TRIM(CAST({sql_column('Commodity')} AS VARCHAR)))
        IN ('WHEAT', 'MAIZE')

    AND TRY_CAST({sql_column('Date')} AS DATE)
        BETWEEN DATE '{START_DATE}'
        AND DATE '{END_DATE}'
"""

print("\nExtracting Maharashtra Wheat and Maize...")

raw = con.execute(query).df()

print("Extracted rows:", len(raw))

if raw.empty:
    raise RuntimeError(
        "No records found. Check the source column names, "
        "commodity labels and state spelling."
    )

# ================================================================
# 8. STANDARDIZE TEXT DATA
# ================================================================

text_columns = [
    "State",
    "District",
    "Market",
    "Commodity",
    "Variety",
    "Grade"
]

for col in text_columns:

    raw[col] = (
        raw[col]
        .astype("string")
        .str.strip()
    )

raw["Commodity"] = raw["Commodity"].str.upper()

raw["District"] = (
    raw["District"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

raw["Market"] = (
    raw["Market"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Normalize common commodity labels.
commodity_mapping = {
    "WHEAT": "WHEAT",
    "MAIZE": "MAIZE",
    "CORN": "MAIZE"
}

raw["Commodity"] = raw["Commodity"].replace(
    commodity_mapping
)

# ================================================================
# 9. CLEAN DATE VARIABLES
# ================================================================

raw["Arrival_Date"] = pd.to_datetime(
    raw["Arrival_Date"],
    errors="coerce"
)

raw = raw.dropna(
    subset=[
        "Arrival_Date",
        "District",
        "Market",
        "Commodity"
    ]
)

raw = raw[
    (raw["Arrival_Date"] >= START_DATE)
    &
    (raw["Arrival_Date"] <= END_DATE)
]

# ================================================================
# 10. CLEAN NUMERIC VARIABLES
# ================================================================

numeric_columns = [
    "Min_Price",
    "Max_Price",
    "Modal_Price",
    "Arrivals"
]

for col in numeric_columns:

    raw[col] = pd.to_numeric(
        raw[col],
        errors="coerce"
    )

# Remove missing or non-positive modal prices.
raw = raw[
    raw["Modal_Price"].notna()
    &
    (raw["Modal_Price"] > 0)
]

# Remove invalid negative prices.
for col in ["Min_Price", "Max_Price"]:

    raw.loc[
        raw[col] <= 0,
        col
    ] = np.nan

# Remove logically inconsistent records.
valid_min = (
    raw["Min_Price"].isna()
    |
    (raw["Min_Price"] <= raw["Modal_Price"])
)

valid_max = (
    raw["Max_Price"].isna()
    |
    (raw["Max_Price"] >= raw["Modal_Price"])
)

raw = raw[valid_min & valid_max]

# Negative arrivals are invalid; missing values remain missing.
raw.loc[
    raw["Arrivals"] < 0,
    "Arrivals"
] = np.nan

# ================================================================
# 11. REMOVE EXACT DUPLICATES
# ================================================================

before = len(raw)

raw = raw.drop_duplicates()

after = len(raw)

print("\nExact duplicates removed:", before - after)

# Save raw filtered data before aggregation.
raw.to_parquet(
    OUTPUT_DIR / "maharashtra_wheat_maize_raw.parquet",
    index=False
)

# ================================================================
# 12. MARKET-LEVEL DAILY AGGREGATION
# ================================================================

# First aggregate variety-level observations within a mandi.
# Arrival-weighted averaging is used only when valid arrivals
# exist for all records contributing to the price estimate.
#
# Otherwise, use an equal-weight mean.

def weighted_or_equal_mean(group, price_col):

    prices = group[price_col]
    arrivals = group["Arrivals"]

    valid = (
        prices.notna()
        &
        arrivals.notna()
        &
        (arrivals > 0)
    )

    if valid.all() and valid.sum() > 0:

        return np.average(
            prices[valid],
            weights=arrivals[valid]
        )

    return prices.mean()


market_records = []

group_cols = [
    "Arrival_Date",
    "State",
    "District",
    "Market",
    "Commodity"
]

for keys, group in tqdm(
    raw.groupby(group_cols, dropna=False),
    desc="Aggregating market data"
):

    date, state, district, market, commodity = keys

    market_records.append({

        "Arrival_Date": date,
        "State": state,
        "District": district,
        "Market": market,
        "Commodity": commodity,

        "mandi_modal_price": weighted_or_equal_mean(
            group, "Modal_Price"
        ),

        "mandi_min_price": group["Min_Price"].mean(),

        "mandi_max_price": group["Max_Price"].mean(),

        "total_arrivals": group["Arrivals"].sum(
            min_count=1
        ),

        "varieties_reported": group["Variety"].nunique(),

        "source_records": len(group)

    })

market_daily = pd.DataFrame(market_records)

market_daily.to_csv(
    OUTPUT_DIR / "maharashtra_market_daily_prices.csv",
    index=False
)

# ================================================================
# 13. DISTRICT-LEVEL DAILY AGGREGATION
# ================================================================

def district_weighted_price(group):

    valid = (
        group["mandi_modal_price"].notna()
        &
        group["total_arrivals"].notna()
        &
        (group["total_arrivals"] > 0)
    )

    if valid.any():

        return np.average(
            group.loc[valid, "mandi_modal_price"],
            weights=group.loc[valid, "total_arrivals"]
        )

    return group["mandi_modal_price"].mean()


district_records = []

district_groups = [
    "Arrival_Date",
    "State",
    "District",
    "Commodity"
]

for keys, group in tqdm(
    market_daily.groupby(district_groups),
    desc="Aggregating districts"
):

    date, state, district, commodity = keys

    district_records.append({

        "Arrival_Date": date,
        "State": state,
        "District": district,
        "Commodity": commodity,

        "district_modal_price": district_weighted_price(
            group
        ),

        "district_avg_min_price": group[
            "mandi_min_price"
        ].mean(),

        "district_avg_max_price": group[
            "mandi_max_price"
        ].mean(),

        "mandi_count": group["Market"].nunique(),

        "mandi_price_sd": group[
            "mandi_modal_price"
        ].std(),

        "lowest_mandi_modal_price": group[
            "mandi_modal_price"
        ].min(),

        "highest_mandi_modal_price": group[
            "mandi_modal_price"
        ].max(),

        "total_arrivals": group[
            "total_arrivals"
        ].sum(min_count=1),

        "total_source_records": group[
            "source_records"
        ].sum()

    })

district_daily = pd.DataFrame(district_records)

# ================================================================
# 14. CREATE TIME FEATURES
# ================================================================

district_daily["Arrival_Date"] = pd.to_datetime(
    district_daily["Arrival_Date"]
)

district_daily["Year"] = (
    district_daily["Arrival_Date"].dt.year
)

district_daily["Month"] = (
    district_daily["Arrival_Date"].dt.month
)

district_daily["Day"] = (
    district_daily["Arrival_Date"].dt.day
)

district_daily["DayOfWeek"] = (
    district_daily["Arrival_Date"].dt.dayofweek
)

district_daily["Quarter"] = (
    district_daily["Arrival_Date"].dt.quarter
)

district_daily["DayOfYear"] = (
    district_daily["Arrival_Date"].dt.dayofyear
)

# Cyclical seasonal features.
district_daily["Month_sin"] = np.sin(
    2 * np.pi * district_daily["Month"] / 12
)

district_daily["Month_cos"] = np.cos(
    2 * np.pi * district_daily["Month"] / 12
)

district_daily["DayOfYear_sin"] = np.sin(
    2 * np.pi * district_daily["DayOfYear"] / 365.25
)

district_daily["DayOfYear_cos"] = np.cos(
    2 * np.pi * district_daily["DayOfYear"] / 365.25
)

# ================================================================
# 15. SORT AND SAVE FINAL DATA
# ================================================================

district_daily = district_daily.sort_values(
    [
        "Commodity",
        "District",
        "Arrival_Date"
    ]
).reset_index(drop=True)

district_daily.to_csv(
    OUTPUT_DIR / "maharashtra_district_daily_prices.csv",
    index=False
)

district_daily.to_parquet(
    OUTPUT_DIR / "maharashtra_district_daily_prices.parquet",
    index=False
)

# ================================================================
# 16. CREATE COMPLETE CALENDAR PANEL
# ================================================================

calendar = pd.date_range(
    START_DATE,
    END_DATE,
    freq="D"
)

districts = sorted(
    district_daily["District"].dropna().unique()
)

index = pd.MultiIndex.from_product(
    [
        districts,
        COMMODITIES,
        calendar
    ],
    names=[
        "District",
        "Commodity",
        "Arrival_Date"
    ]
)

panel = (
    district_daily
    .set_index(
        [
            "District",
            "Commodity",
            "Arrival_Date"
        ]
    )
    .reindex(index)
    .reset_index()
)

panel["price_observed"] = (
    panel["district_modal_price"]
    .notna()
    .astype(int)
)

# Do not fill missing target prices with zero or forward-fill.
panel.to_csv(
    OUTPUT_DIR / "maharashtra_complete_daily_panel.csv",
    index=False
)

# ================================================================
# 17. QUALITY CONTROL REPORT
# ================================================================

report = {}

report["raw_records"] = int(len(raw))

report["market_daily_records"] = int(
    len(market_daily)
)

report["district_daily_records"] = int(
    len(district_daily)
)

report["districts"] = int(
    district_daily["District"].nunique()
)

report["commodities"] = (
    district_daily["Commodity"].unique().tolist()
)

report["date_min"] = str(
    district_daily["Arrival_Date"].min()
)

report["date_max"] = str(
    district_daily["Arrival_Date"].max()
)

report["missing_modal_prices"] = int(
    district_daily["district_modal_price"].isna().sum()
)

report["duplicate_district_days"] = int(
    district_daily.duplicated(
        [
            "District",
            "Commodity",
            "Arrival_Date"
        ]
    ).sum()
)

print("\n" + "=" * 65)
print("FINAL DATA QUALITY REPORT")
print("=" * 65)

for key, value in report.items():
    print(f"{key}: {value}")

# ================================================================
# 18. YEAR-WISE COVERAGE
# ================================================================

coverage = (
    district_daily.groupby(
        ["Commodity", "Year"]
    )
    .agg(
        observations=("district_modal_price", "count"),
        districts=("District", "nunique"),
        average_mandis=("mandi_count", "mean"),
        average_price=("district_modal_price", "mean")
    )
    .reset_index()
)

coverage.to_csv(
    OUTPUT_DIR / "yearwise_coverage_report.csv",
    index=False
)

# ================================================================
# 19. MISSING DATA REPORT
# ================================================================

missing_report = (
    panel.groupby(
        ["Commodity", "District"]
    )
    .agg(
        expected_days=("Arrival_Date", "count"),
        observed_days=("price_observed", "sum")
    )
    .reset_index()
)

missing_report["missing_days"] = (
    missing_report["expected_days"]
    - missing_report["observed_days"]
)

missing_report["coverage_percent"] = (
    100 * missing_report["observed_days"]
    / missing_report["expected_days"]
)

missing_report.to_csv(
    OUTPUT_DIR / "missing_data_report.csv",
    index=False
)

# ================================================================
# 20. SAVE FINAL EXECUTION LOG
# ================================================================

with open(
    LOG_DIR / "execution_summary.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report,
        f,
        indent=4
    )

print("\n" + "=" * 65)
print("EXTRACTION AND CLEANING COMPLETED")
print("=" * 65)

print("Final district records:", len(district_daily))
print("Final panel records:", len(panel))

print("\nSaved files:")

for file in sorted(OUTPUT_DIR.glob("*")):
    if file.is_file():
        print(file.name)

print("\nOutput directory:")
print(OUTPUT_DIR)

con.close()


PROJECT CONFIGURATION
Base directory: C:\Users\DELL\Documents\PROJECT 4
Output directory: C:\Users\DELL\Documents\PROJECT 4\output\mandi_prices
Download directory: C:\Users\DELL\Documents\PROJECT 4\output\mandi_prices\raw

Connecting to dataset repository...
Total repository files: 29
Parquet files: 26


data/mandi_daily_by_year/mandi_daily_200(…): reconstructing file:   0%|          |  0.00B /  128kB            

data/mandi_daily_by_year/mandi_daily_200(…): downloading bytes:           |  0.00B            

data/mandi_daily_by_year/mandi_daily_200(…): reconstructing file:   0%|          |  0.00B / 2.98MB            

data/mandi_daily_by_year/mandi_daily_200(…): downloading bytes:           |  0.00B            

data/mandi_daily_by_year/mandi_daily_200(…): reconstructing file:   0%|          |  0.00B / 8.63MB            

... (29 outputs left out)

data/mandi_daily_by_year/mandi_daily_201(…): reconstructing file:   0%|          |  0.00B / 47.7MB            

data/mandi_daily_by_year/mandi_daily_201(…): downloading bytes:           |  0.00B            

data/mandi_daily_by_year/mandi_daily_201(…): reconstructing file:   0%|          |  0.00B / 47.9MB            

data/mandi_daily_by_year/mandi_daily_201(…): downloading bytes:           |  0.00B            